<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [ ]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [ ]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

    !wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
    !wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
    !wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
    !wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

In [ ]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [ ]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [ ]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---

LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
IMAGE_DIR = "/content/image_dir"

In [ ]:
%cd /content/

/content


### ✅ Packages Handling

In [ ]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [ ]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched, collate_fn
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [ ]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [ ]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = False # @param {type:"boolean"}


In [ ]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [ ]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [ ]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
if dnzd_pw == False:
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
else:
    dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True, collate_fn=collate_fn)

In [ ]:
shape = None
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    shape = i5.shape
    break

In [ ]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [ ]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:28<00:00,  2.91it/s]


In [ ]:
processed_micrographs.shape

(84, 3840, 3840)

In [ ]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [ ]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [ ]:
del model
torch.cuda.empty_cache()

In [ ]:
model = model_post

In [ ]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF'

In [ ]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()

        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [ ]:
!cp {RESULT_DIR}/label_images.npy .

In [ ]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [ ]:
label_images.shape

(84, 3840, 3840)

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [ ]:
!cp {RESULT_DIR}/best_config.txt .

In [ ]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

---

In [ ]:
!cp {RESULT_DIR}/best_watershed_params.json .

In [ ]:
import json

# Load the best parameters found
with open('best_watershed_params.json', 'r') as f:
    best_params = json.load(f)

# changed) Extracting specific values from the parsed dictionary
best_thresh = best_params['thresh']
best_dist = best_params['dist']

print(f"Loaded Config: Thresh={best_thresh}, Dist={best_dist}")

ws_config = [best_thresh, best_dist]

Loaded Config: Thresh=0.4, Dist=0.8


---

In [ ]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import starfile

---

In [ ]:
def calculate_shape_metrics(region):
    """Computes Sphericity, Circularity, and Inverse Slenderness."""
    area = region.area
    perimeter = region.perimeter
    if perimeter == 0: return 0, 0, 0

    circularity = (4 * np.pi * area) / (perimeter ** 2)

    d_eq = np.sqrt(4 * area / np.pi)
    d_cir = region.major_axis_length
    sphericity = d_eq / d_cir if d_cir > 0 else 0

    major = region.major_axis_length
    minor = region.minor_axis_length
    inv_slenderness = minor / major if major > 0 else 0

    return sphericity, circularity, inv_slenderness

In [ ]:
def calculate_shape_metrics(region):
        """Calculates custom descriptors and first 3 Hu Moments."""

        def _circularity_(region): return (4 * np.pi * region.area) / (region.perimeter ** 2) if region.perimeter > 0 else 0

        def _eccentricity_(region): return region.eccentricity

        def _solidity_(region): return region.area / region.convex_area if region.convex_area > 0 else 0

        base_metrics = [_circularity_(region), _eccentricity_(region), _solidity_(region)]

        hu_moments = list(region.moments_hu[:3])

        return tuple(base_metrics + hu_moments), region.area

In [ ]:
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

def detect_particles_dt(prob_map_input, particle_radius=64,
                        peak_threshold_ratio=0.6, min_dist_ratio=0.4,
                        show_plots=True):
    # 1. Normalization
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]
    if prob_map.max() > 1.0: prob_map /= 255.0

    binary_mask = (prob_map > 0.5).astype(np.uint8)
    if np.sum(binary_mask) == 0: binary_mask = (prob_map > 0.05).astype(np.uint8)
    if np.sum(binary_mask) == 0: return []

    # 2. DT transform
    distance = ndi.distance_transform_edt(binary_mask)

    # 3. Markers Finding
    min_dist = int(particle_radius * min_dist_ratio)
    min_peak_height = particle_radius * peak_threshold_ratio

    local_max_coords = peak_local_max(
        distance,
        min_distance=min_dist,
        labels=binary_mask,
        threshold_abs=min_peak_height
    )

    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 4. Watershed
    labels = watershed(-distance, markers, mask=binary_mask)

    # 5. Extract Candidate Metrics
    expected_area = np.pi * (particle_radius ** 2)
    min_area, max_area = expected_area * 0.3, expected_area * 2.0

    regions = regionprops(labels)
    candidates = []
    m_sp, m_cr, m_isd = [], [], []

    for region in regions:
        if min_area <= region.area <= max_area:
            sp, cr, isd = calculate_shape_metrics(region)
            candidates.append({'region': region, 'metrics': (sp, cr, isd)})
            m_sp.append(sp); m_cr.append(cr); m_isd.append(isd)

    if not candidates: return []

    # 6. Statistical Thresholding (Modified Z-Score / MAD)
    #  MAD statistics thresholding
    def get_mad_range(data, z_thresh=3.0):
        if not data: return 0.0, 1.0
        arr = np.array(data)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad == 0: return np.min(arr), np.max(arr)

        # Calculate bounds based on 3 median absolute deviations
        low = med - (z_thresh * mad / 0.6745)
        high = med + (z_thresh * mad / 0.6745)
        return max(0.0, low), min(1.0, high)

    thresh_sp = get_mad_range(m_sp)
    thresh_cr = get_mad_range(m_cr)
    thresh_isd = get_mad_range(m_isd)

    # 7. Auto-Visualization
    if show_plots:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        metrics = [m_sp, m_cr, m_isd]
        titles = ['Sphericity', 'Circularity', 'Inv-Slenderness']
        bounds = [thresh_sp, thresh_cr, thresh_isd]

        for i in range(3):
            axes[i].hist(metrics[i], bins=20, color='gray', alpha=0.5)
            axes[i].axvline(bounds[i][0], color='red', linestyle='--', label='Min')
            axes[i].axvline(bounds[i][1], color='blue', linestyle='--', label='Max')
            axes[i].set_title(titles[i])
            if i == 0: axes[i].legend()
        plt.suptitle(f"Adaptive Thresholds for Current Image (N={len(candidates)})")
        plt.show()

    # 8. Final Selection
    valid_particles = []
    boundary_margin = 40
    for cand in candidates:
        sp, cr, isd = cand['metrics']
        if (thresh_sp[0] <= sp <= thresh_sp[1] and
            thresh_cr[0] <= cr <= thresh_cr[1] and
            thresh_isd[0] <= isd <= thresh_isd[1]):

            region = cand['region']
            cy, cx = region.centroid
            if (boundary_margin < cx < width - boundary_margin and
                boundary_margin < cy < height - boundary_margin):
                score = prob_map[int(cy), int(cx)]
                valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
radius = RADIUS

watershed_config = ws_config
watershed_list = []

for img in label_images:
    particles = detect_particles_dt(
        img,
        particle_radius=radius,
        peak_threshold_ratio=watershed_config[0],
        min_dist_ratio=watershed_config[1]
    )
    watershed_list.append(particles)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import matplotlib

centre_cord_List = watershed_list

for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in centre_cord_List[idx]:
            x, y, _ = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y, _ in centre_cord_List[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,3354,802,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,549,1206,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1771,418,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,2265,843,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,217,2979,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
40232,3325,2433,00425@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40233,3175,1867,00426@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40234,817,1316,00427@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40235,1049,774,00428@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [ ]:
starfile.write(df, f'{RESULT_DIR}/watershed_particles.star')

---
Analysis on the filtered particles

In [ ]:
def detect_particles_dt_filtered(prob_map_input, particle_radius=64,
                                 peak_threshold_ratio=0.6, min_dist_ratio=0.4,
                                 show_plots=False):
    # --- 1-4. Preprocessing & Watershed (Same as before) ---
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]
    if prob_map.max() > 1.0: prob_map /= 255.0
    binary_mask = (prob_map > 0.5).astype(np.uint8)
    if np.sum(binary_mask) == 0: binary_mask = (prob_map > 0.05).astype(np.uint8)
    if np.sum(binary_mask) == 0: return {"valid": [], "area_filtered": [], "shape_filtered": []}

    distance = ndi.distance_transform_edt(binary_mask)
    local_max_coords = peak_local_max(distance, min_distance=int(particle_radius * min_dist_ratio),
                                      labels=binary_mask, threshold_abs=particle_radius * peak_threshold_ratio)
    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords): markers[r, c] = i + 1
    labels = watershed(-distance, markers, mask=binary_mask)

    # --- 5. Initial Area Filtering ---
    expected_area = np.pi * (particle_radius ** 2)
    min_area, max_area = expected_area * 0.3, expected_area * 2.0
    regions = regionprops(labels)

    candidates = []
    area_filtered = []
    m_sp, m_cr, m_isd = [], [], []

    for region in regions:
        cy, cx = region.centroid
        if not (min_area <= region.area <= max_area):
            # changed) Collect particles rejected by size
            area_filtered.append([int(cx), int(cy)])
            continue

        sp, cr, isd = calculate_shape_metrics(region)
        candidates.append({'region': region, 'metrics': (sp, cr, isd)})
        m_sp.append(sp); m_cr.append(cr); m_isd.append(isd)

    if not candidates:
        return {"valid": [], "area_filtered": area_filtered, "shape_filtered": []}

    # --- 6. Robust Adaptive Thresholding (MAD) ---
    def get_mad_range(data, z_thresh=3.0):
        arr = np.array(data)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad == 0: return np.min(arr), np.max(arr)
        low = med - (z_thresh * mad / 0.6745)
        high = med + (z_thresh * mad / 0.6745)
        return max(0.0, low), min(1.0, high)

    t_sp = get_mad_range(m_sp)
    t_cr = get_mad_range(m_cr)
    t_isd = get_mad_range(m_isd)

    # --- 8. Final Categorization ---
    valid = []
    shape_filtered = []
    boundary_margin = 40

    for cand in candidates:
        sp, cr, isd = cand['metrics']
        region = cand['region']
        cy, cx = region.centroid

        # Check Adaptive Thresholds
        if (t_sp[0] <= sp <= t_sp[1] and
            t_cr[0] <= cr <= t_cr[1] and
            t_isd[0] <= isd <= t_isd[1]):

            # Check Boundary
            if (boundary_margin < cx < width - boundary_margin and
                boundary_margin < cy < height - boundary_margin):
                valid.append([int(cx), int(cy), float(prob_map[int(cy), int(cx)])])
            else:
                area_filtered.append([int(cx), int(cy)]) # Boundary rejections treated as area/loc rejection
        else:
            # changed) Collect particles rejected by shape
            shape_filtered.append([int(cx), int(cy)])

    return {
        "valid": valid,
        "area_filtered": area_filtered,
        "shape_filtered": shape_filtered
    }

In [ ]:
def plot_micrograph_filtered(ax, micrograph, labels, particle_dict):
    """
    Plots the results using centered crosses:
    - Red '+': Valid Detections
    - Royalblue 'x': Area Rejections
    - Navy 'x': Shape Rejections
    """
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='inferno', alpha=0.4)

    # 1. Plot Area Filtered (Royalblue Cross)
    if particle_dict['area_filtered']:
        area_pts = np.array(particle_dict['area_filtered'])
        ax.scatter(area_pts[:, 0], area_pts[:, 1], marker='x', color='limegreen',
                   s=100, linewidths=2, label='Area Rejected')

    # 2. Plot Shape Filtered (Navy Cross)
    if particle_dict['shape_filtered']:
        shape_pts = np.array(particle_dict['shape_filtered'])
        ax.scatter(shape_pts[:, 0], shape_pts[:, 1], marker='x', color='navy',
                   s=100, linewidths=2, label='Shape Rejected')

    # 3. Plot Valid (Red Plus)
    if particle_dict['valid']:
        valid_pts = np.array(particle_dict['valid'])
        ax.scatter(valid_pts[:, 0], valid_pts[:, 1], marker='+', color='red',
                   s=150, linewidths=1.5, label='Valid')

    # Handle Legend (prevent duplicate labels)
    handles, labels_lg = ax.get_legend_handles_labels()
    by_label = dict(zip(labels_lg, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True)

In [ ]:
radius = RADIUS

watershed_config = ws_config
watershed_list_err_analy = []

for img in label_images:
    particles = detect_particles_dt_filtered(
        img,
        particle_radius=radius,
        peak_threshold_ratio=watershed_config[0],
        min_dist_ratio=watershed_config[1]
    )
    watershed_list_err_analy.append(particles)

In [ ]:
centre_cord_List = watershed_list_err_analy

for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    if idx % 30 == 0:

        label_image = label_images[idx]

        fig, ax = plt.subplots(figsize=(12, 12))

        particle_dict = centre_cord_List[idx]

        plot_micrograph_filtered(ax, processed_micrographs[idx], label_image, particle_dict)

        plt.show()

Output hidden; open in https://colab.research.google.com to view.

---